SCENARIO: “University Smart Assistant with Role-Based Access”
🏫 Background Story
A university deploys an AI-powered academic assistant to help students, faculty, and administrators.
👉 Users can ask:
- “What is my attendance record?”
- “Show me my exam results.”
- “Update course schedules.”
- “Approve new course registrations.”
👉 But not everyone can do everything — access depends on roles.

In [ ]:
!pip install groq gradio nest_asyncio

import os
import asyncio
import nest_asyncio
import gradio as gr
from groq import Groq

os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

nest_asyncio.apply()

_original_asyncio_run = asyncio.run

def compatible_asyncio_run(main, *, debug=None, loop_factory=None):
    return _original_asyncio_run(main)

asyncio.run = compatible_asyncio_run


users = {
    "S101": {
        "role": "student",
        "name": "Rahul Sharma",
        "attendance": "88%",
        "exam_results": {
            "DBMS": "A",
            "OS": "B+",
            "CN": "A-"
        },
        "courses": ["DBMS", "OS", "CN", "AI"]
    },
    "S102": {
        "role": "student",
        "name": "Priya Verma",
        "attendance": "93%",
        "exam_results": {
            "DBMS": "A+",
            "OS": "A",
            "CN": "B+"
        },
        "courses": ["DBMS", "OS", "CN", "ML"]
    },
    "F201": {
        "role": "faculty",
        "name": "Dr. Mehta",
        "own_attendance_access_note": "Faculty self attendance available in system logs",
        "faculty_schedule": [
            {"day": "Monday", "time": "10:00 AM", "course": "DBMS"},
            {"day": "Tuesday", "time": "12:00 PM", "course": "OS"},
            {"day": "Thursday", "time": "02:00 PM", "course": "AI"}
        ],
        "assigned_courses": ["DBMS", "OS", "AI"],
        "assigned_students": ["Rahul Sharma", "Priya Verma", "Aman Singh"]
    },
    "F202": {
        "role": "faculty",
        "name": "Dr. Khanna",
        "own_attendance_access_note": "Faculty self attendance available in system logs",
        "faculty_schedule": [
            {"day": "Wednesday", "time": "11:00 AM", "course": "CN"},
            {"day": "Friday", "time": "01:00 PM", "course": "ML"}
        ],
        "assigned_courses": ["CN", "ML"],
        "assigned_students": ["Neha Jain", "Priya Verma"]
    },
    "A301": {
        "role": "admin",
        "name": "Admin Office",
        "own_attendance_access_note": "Admin attendance available in staff control records",
        "pending_registrations": [
            {"student": "Rahul Sharma", "course": "Cloud Computing", "status": "Pending"},
            {"student": "Neha Jain", "course": "Cyber Security", "status": "Pending"}
        ],
        "course_schedule_updates": [
            {"course": "DBMS", "old_time": "10:00 AM", "new_time": "11:00 AM", "status": "Draft"},
            {"course": "ML", "old_time": "01:00 PM", "new_time": "02:30 PM", "status": "Draft"}
        ],
        "approval_queue": [
            {"request_id": "R001", "type": "Course Registration", "status": "Pending"},
            {"request_id": "R002", "type": "Schedule Change", "status": "Pending"}
        ]
    }
}


student_master_records = {
    "Rahul Sharma": {
        "attendance": "88%",
        "exam_results": {"DBMS": "A", "OS": "B+", "CN": "A-"},
        "courses": ["DBMS", "OS", "CN", "AI"]
    },
    "Priya Verma": {
        "attendance": "93%",
        "exam_results": {"DBMS": "A+", "OS": "A", "CN": "B+"},
        "courses": ["DBMS", "OS", "CN", "ML"]
    },
    "Aman Singh": {
        "attendance": "81%",
        "exam_results": {"DBMS": "B", "OS": "B", "CN": "B+"},
        "courses": ["DBMS", "CN"]
    },
    "Neha Jain": {
        "attendance": "90%",
        "exam_results": {"DBMS": "A", "OS": "A-", "CN": "A"},
        "courses": ["CN", "ML", "Cyber Security"]
    }
}


permissions = {
    "student": [
        "view_own_attendance"
    ],
    "faculty": [
        "view_own_attendance",
        "view_others_attendance",
        "update_exam_results",
        "view_all_student_records"
    ],
    "admin": [
        "view_own_attendance",
        "view_others_attendance",
        "update_exam_results",
        "approve_course_changes",
        "view_all_student_records"
    ]
}


intent_permission_map = {
    "own_attendance": "view_own_attendance",
    "others_attendance": "view_others_attendance",
    "exam_results_update": "update_exam_results",
    "approve_course_changes": "approve_course_changes",
    "student_records": "view_all_student_records",
    "full_summary": None
}


async def get_own_attendance(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role == "student":
        return (
            f"Own Attendance Record:\n"
            f"- Name: {users[user_id]['name']}\n"
            f"- Role: Student\n"
            f"- Attendance: {users[user_id]['attendance']}"
        )

    if role == "faculty":
        return (
            f"Own Attendance Record:\n"
            f"- Name: {users[user_id]['name']}\n"
            f"- Role: Faculty\n"
            f"- Note: {users[user_id]['own_attendance_access_note']}"
        )

    if role == "admin":
        return (
            f"Own Attendance Record:\n"
            f"- Name: {users[user_id]['name']}\n"
            f"- Role: Admin\n"
            f"- Note: {users[user_id]['own_attendance_access_note']}"
        )

    return "Invalid user."


async def get_others_attendance(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role not in ["faculty", "admin"]:
        return "Access denied."

    text = "Others' Attendance Records:\n"
    for student_name, data in student_master_records.items():
        text += f"- {student_name}: {data['attendance']}\n"
    return text.strip()


async def update_exam_results_data(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role not in ["faculty", "admin"]:
        return "Access denied."

    text = (
        "Exam Result Update Access:\n"
        f"- User: {users[user_id]['name']}\n"
        f"- Role: {role.title()}\n"
        "- Status: Authorized to update exam results\n"
        "- Scope: Can modify student marks/grades in academic records"
    )
    return text


async def get_all_student_records(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role not in ["faculty", "admin"]:
        return "Access denied."

    text = "All Student Records:\n"
    for student_name, data in student_master_records.items():
        results_text = ", ".join([f"{sub}: {grade}" for sub, grade in data["exam_results"].items()])
        courses_text = ", ".join(data["courses"])
        text += (
            f"- Name: {student_name}\n"
            f"  Attendance: {data['attendance']}\n"
            f"  Results: {results_text}\n"
            f"  Courses: {courses_text}\n"
        )
    return text.strip()


async def approve_course_changes_data(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role != "admin":
        return "Access denied."

    pending_regs = users[user_id]["pending_registrations"]
    schedule_updates = users[user_id]["course_schedule_updates"]
    approval_queue = users[user_id]["approval_queue"]

    text = f"Course Change and Approval Access:\n- Admin: {users[user_id]['name']}\n\n"
    text += "Pending Registrations:\n"
    for item in pending_regs:
        text += f"- Student: {item['student']}, Course: {item['course']}, Status: {item['status']}\n"

    text += "\nSchedule Update Requests:\n"
    for item in schedule_updates:
        text += (
            f"- Course: {item['course']}, Old Time: {item['old_time']}, "
            f"New Time: {item['new_time']}, Status: {item['status']}\n"
        )

    text += "\nApproval Queue:\n"
    for item in approval_queue:
        text += f"- Request ID: {item['request_id']}, Type: {item['type']}, Status: {item['status']}\n"

    return text.strip()


async def parallel_role_data_fetch(user_id):
    role = users[user_id]["role"]

    if role == "student":
        results = await asyncio.gather(
            get_own_attendance(user_id),
            return_exceptions=True
        )
        return {
            "own_attendance": results[0] if not isinstance(results[0], Exception) else "Own attendance unavailable"
        }

    if role == "faculty":
        results = await asyncio.gather(
            get_own_attendance(user_id),
            get_others_attendance(user_id),
            update_exam_results_data(user_id),
            get_all_student_records(user_id),
            return_exceptions=True
        )
        return {
            "own_attendance": results[0] if not isinstance(results[0], Exception) else "Own attendance unavailable",
            "others_attendance": results[1] if not isinstance(results[1], Exception) else "Others attendance unavailable",
            "exam_results_update": results[2] if not isinstance(results[2], Exception) else "Exam result update access unavailable",
            "student_records": results[3] if not isinstance(results[3], Exception) else "Student records unavailable"
        }

    if role == "admin":
        results = await asyncio.gather(
            get_own_attendance(user_id),
            get_others_attendance(user_id),
            update_exam_results_data(user_id),
            approve_course_changes_data(user_id),
            get_all_student_records(user_id),
            return_exceptions=True
        )
        return {
            "own_attendance": results[0] if not isinstance(results[0], Exception) else "Own attendance unavailable",
            "others_attendance": results[1] if not isinstance(results[1], Exception) else "Others attendance unavailable",
            "exam_results_update": results[2] if not isinstance(results[2], Exception) else "Exam result update access unavailable",
            "approve_course_changes": results[3] if not isinstance(results[3], Exception) else "Approval data unavailable",
            "student_records": results[4] if not isinstance(results[4], Exception) else "Student records unavailable"
        }

    return {}


def decide_intent(user_query, role):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
You are a university smart assistant intent classifier.

Current role: {role}

Allowed intents for student:
- own_attendance
- full_summary

Allowed intents for faculty:
- own_attendance
- others_attendance
- exam_results_update
- student_records
- full_summary

Allowed intents for admin:
- own_attendance
- others_attendance
- exam_results_update
- approve_course_changes
- student_records
- full_summary

Rules:
- "What is my attendance record?" -> own_attendance
- "Show others attendance" -> others_attendance
- "Update exam results" -> exam_results_update
- "Approve new course registrations" or "Approve course changes" -> approve_course_changes
- "View all student records" -> student_records
- "Give me full summary" -> full_summary

Return exactly one valid intent for the current role.
If something is not permitted for that role, still choose the closest intent so the system can deny access properly.

User Query: {user_query}
"""
        }]
    )
    return response.choices[0].message.content.strip().lower()


def analyse_university_data(text, role):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Analyze this university RBAC-based data for a {role} user and provide:
1. Key summary
2. Important observations
3. Role-based access note
4. Operational or academic insight
5. Simple user-friendly explanation

Data:
{text}
"""
        }]
    )
    return response.choices[0].message.content


def generate_university_report(analysis, user_id):
    user_name = users.get(user_id, {}).get("name", "User")
    user_role = users.get(user_id, {}).get("role", "unknown")

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Create a professional but simple university smart assistant report for {user_name}.

Role: {user_role}

Use this analysis:
{analysis}

Keep it:
- clear
- structured
- role-aware
- concise but informative
"""
        }]
    )
    return response.choices[0].message.content


async def full_pipeline(user_id, user_query):
    if user_id not in users:
        return "Invalid User ID. Please enter S101, S102, F201, F202, or A301."

    role = users[user_id]["role"]
    intent = decide_intent(user_query, role)
    data = await parallel_role_data_fetch(user_id)

    required_permission = intent_permission_map.get(intent)

    if required_permission is not None and required_permission not in permissions[role]:
        return (
            "==============================\n"
            "UNIVERSITY SMART ASSISTANT OUTPUT\n"
            "==============================\n\n"
            f"User ID: {user_id}\n"
            f"Role: {role}\n"
            f"Detected Intent: {intent}\n\n"
            f"Access Denied: {role} cannot perform '{required_permission}'."
        )

    if role == "student":
        if intent == "own_attendance":
            combined_text = data["own_attendance"]
        elif intent == "full_summary":
            combined_text = "\n\n".join(data.values())
        else:
            return (
                "==============================\n"
                "UNIVERSITY SMART ASSISTANT OUTPUT\n"
                "==============================\n\n"
                f"User ID: {user_id}\n"
                f"Role: {role}\n"
                f"Detected Intent: {intent}\n\n"
                "Access Denied: Student role has limited permissions based on RBAC policy."
            )

    elif role == "faculty":
        if intent == "own_attendance":
            combined_text = data["own_attendance"]
        elif intent == "others_attendance":
            combined_text = data["others_attendance"]
        elif intent == "exam_results_update":
            combined_text = data["exam_results_update"]
        elif intent == "student_records":
            combined_text = data["student_records"]
        else:
            combined_text = "\n\n".join(data.values())

    elif role == "admin":
        if intent == "own_attendance":
            combined_text = data["own_attendance"]
        elif intent == "others_attendance":
            combined_text = data["others_attendance"]
        elif intent == "exam_results_update":
            combined_text = data["exam_results_update"]
        elif intent == "approve_course_changes":
            combined_text = data["approve_course_changes"]
        elif intent == "student_records":
            combined_text = data["student_records"]
        else:
            combined_text = "\n\n".join(data.values())

    else:
        return "Invalid role."

    analysis = analyse_university_data(combined_text, role)
    report = generate_university_report(analysis, user_id)

    final_output = f"""
==============================
UNIVERSITY SMART ASSISTANT OUTPUT
==============================

User ID: {user_id}
Role: {role}
Detected Intent: {intent}

RAW DATA:
{combined_text}

--------------------------------
AI ANALYSIS + ROLE-BASED REPORT:
--------------------------------
{report}
"""
    return final_output.strip()


def run_normal_mode_logic(user_id, user_question):
    user_id = user_id.strip()
    user_question = user_question.strip()

    if not user_id or not user_question:
        return "Please enter both User ID and question."

    return asyncio.run(full_pipeline(user_id, user_question))


def university_assistant_ui(user_id, user_query):
    user_id = user_id.strip()
    user_query = user_query.strip()

    if not user_id or not user_query:
        return "Please enter both User ID and question."

    return asyncio.run(full_pipeline(user_id, user_query))


with gr.Blocks() as demo:
    gr.Markdown("# University Smart Assistant with Role-Based Access")
    gr.Markdown("""
Use IDs by role:

Student IDs:
- S101
- S102

Faculty IDs:
- F201
- F202

Admin ID:
- A301

Example questions:
- What is my attendance record?
- Show others attendance.
- Update exam results.
- Approve new course registrations.
- View all student records.
- Give me full summary.
""")

    with gr.Tab("Normal Input Mode"):
        normal_user_id = gr.Textbox(
            label="Enter User ID",
            placeholder="Example: S101 or F201 or A301"
        )
        normal_query = gr.Textbox(
            label="Ask your question",
            placeholder="Example: What is my attendance record?"
        )
        normal_output = gr.Textbox(
            label="Final Output",
            lines=24
        )
        normal_btn = gr.Button("Run Normal Mode")
        normal_btn.click(
            fn=run_normal_mode_logic,
            inputs=[normal_user_id, normal_query],
            outputs=normal_output
        )

    with gr.Tab("Gradio Real-Time Mode"):
        user_id_input = gr.Textbox(
            label="Enter User ID",
            placeholder="Example: S101 or F201 or A301"
        )

        query_input = gr.Textbox(
            label="Ask your question",
            placeholder="Example: What is my attendance record?"
        )

        output_box = gr.Textbox(
            label="Assistant Response",
            lines=24
        )

        submit_btn = gr.Button("Get Details")
        submit_btn.click(
            fn=university_assistant_ui,
            inputs=[user_id_input, query_input],
            outputs=output_box
        )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://018fdd621827253150.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
